# Hello Ray — Distributed Compute on the Playground

Connects to the in-cluster **RayCluster** (deployed via KubeRay) and runs distributed tasks.

**Prerequisite:** the Ray component is installed (`components/ray/install.sh`).

> ⚠️ The Ray Client version below **must match** the cluster's Ray version (2.52.0).
> If you hit a version-mismatch error, check the cluster version and re-pin:
> `kubectl get pod -n ray -l ray.io/node-type=head -o jsonpath='{.items[0].spec.containers[0].image}'`

In [ ]:
# Install a Ray Client matching the cluster (2.52.0)
!pip install -q "ray[client]==2.52.0"

In [ ]:
import ray

# The head service is reachable cross-namespace at <svc>.<namespace>:<port>
ray.init(address="ray://raycluster-kuberay-head-svc.ray:10001")

print("Connected to Ray!")
print("Cluster resources:", ray.cluster_resources())

## Run a distributed task

`@ray.remote` turns a function into a task that runs on any node in the cluster.
We launch several in parallel and collect the results.

In [ ]:
import time
import socket

@ray.remote
def where_am_i(i):
    """Return which pod/host actually ran this task."""
    time.sleep(1)
    return f"task {i} ran on {socket.gethostname()}"

# Launch 8 tasks in parallel across the cluster
futures = [where_am_i.remote(i) for i in range(8)]
results = ray.get(futures)
for r in results:
    print(r)

You should see tasks spread across the **head** and **worker** pods — that's distributed execution.

## A tiny parallel compute

Estimate π with a Monte Carlo simulation, sharded across the cluster.

In [ ]:
import random

@ray.remote
def sample(n):
    inside = 0
    for _ in range(n):
        x, y = random.random(), random.random()
        if x * x + y * y <= 1.0:
            inside += 1
    return inside

SHARDS = 8
PER_SHARD = 1_000_000
counts = ray.get([sample.remote(PER_SHARD) for _ in range(SHARDS)])
pi = 4 * sum(counts) / (SHARDS * PER_SHARD)
print(f"Estimated π ≈ {pi}")

In [ ]:
# Disconnect the client (does NOT shut down the cluster)
ray.shutdown()
print("Disconnected. The RayCluster keeps running for the next notebook.")